# Results Analysis

Metrics, scatter plots, gap correlation and SIMON reproducibility.

| File | N | Description |
|------|---|-------------|
| `ixi_faceage_splits.csv` | 476 | IXI predictions with train/val/test split |
| `ixi_faceage_multiview_raw.csv` | 406 | FaceAge DL 9-view, raw (pre-calibration) |
| `ixi_brainage_synthba.csv` | 105 | SynthBA brain-age on IXI |
| `ixi_gap_correlation.csv` | 93 | Face + brain age gap merged |
| `simon_faceage_morphometrics.csv` | 107 | GPA+Ridge predictions on SIMON |
| `simon_brainage_synthba.csv` | 99 | SynthBA brain-age on SIMON |


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

ROOT = next(
    p for p in [Path.cwd(), Path.cwd().parent]
    if (p / "papers" / "tables").exists()
)
TABLES = ROOT / "papers" / "tables"
FIGURES = ROOT / "papers" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})
print("ROOT:", ROOT)
print("Tables found:", sorted(p.name for p in TABLES.glob("*.csv")))


In [ ]:
# Load all tables
face_ixi    = pd.read_csv(TABLES / "ixi_faceage_splits.csv")
brain_ixi   = pd.read_csv(TABLES / "ixi_brainage_synthba.csv").rename(
    columns={"chron_age": "true_age"}
)
# gap_correlation has a trailing SUMMARY row with extra fields — skip bad lines
gap_df = pd.read_csv(TABLES / "ixi_gap_correlation.csv", on_bad_lines="skip")
gap_df = gap_df[pd.to_numeric(gap_df["ixi_num"], errors="coerce").notna()].copy()

face_simon  = pd.read_csv(TABLES / "simon_faceage_morphometrics.csv")
brain_simon = pd.read_csv(TABLES / "simon_brainage_synthba.csv").rename(
    columns={"chron_age": "true_age"}
)

assert not face_ixi.empty,    "face_ixi empty"
assert not brain_ixi.empty,   "brain_ixi empty"
assert not gap_df.empty,      "gap_df empty"
assert not face_simon.empty,  "face_simon empty"
assert not brain_simon.empty, "brain_simon empty"

for name, df in [
    ("face_ixi", face_ixi), ("brain_ixi", brain_ixi), ("gap_df", gap_df),
    ("face_simon", face_simon), ("brain_simon", brain_simon),
]:
    print(f"{name:20s}: {df.shape}  cols={list(df.columns)}")


In [ ]:
def compute_metrics(df, pred="predicted_age", true="true_age", label=""):
    d = df[[true, pred]].dropna()
    assert len(d) > 5, f"Too few rows in {label!r}: {len(d)}"
    err = d[pred] - d[true]
    r, p = stats.pearsonr(d[true], d[pred])
    return {
        "Method": label,
        "N": len(d),
        "MAE":  round(float(err.abs().mean()), 2),
        "RMSE": round(float(np.sqrt((err**2).mean())), 2),
        "Bias": round(float(err.mean()), 2),
        "r":    round(float(r), 3),
        "p":    f"{p:.2e}",
    }

face_test = face_ixi[face_ixi["split"] == "test"]
assert len(face_test) > 0, "No test-split rows in face_ixi"

rows = [
    compute_metrics(face_test, label="ixi_faceage_splits  (test)"),
    compute_metrics(face_ixi,  label="ixi_faceage_splits  (all)"),
    # morphometrics results from README — no separate CSV yet
    {"Method": "Face morphometrics Ridge (test)",
     "N": 72, "MAE": 7.81, "RMSE": 9.77, "Bias": -1.83, "r": 0.775, "p": "<0.001"},
    compute_metrics(brain_ixi, label="SynthBA IXI"),
]

summary = pd.DataFrame(rows)
pd.set_option("display.float_format", "{:.2f}".format)
print(summary.to_string(index=False))
summary


In [ ]:
def scatter_ax(ax, df, title, color, pred="predicted_age", true="true_age"):
    d = df[[true, pred]].dropna()
    err = d[pred] - d[true]
    mae = float(err.abs().mean())
    r, _ = stats.pearsonr(d[true], d[pred])
    ax.scatter(d[true], d[pred], alpha=0.5, s=20, color=color)
    lo = min(d[true].min(), d[pred].min()) - 2
    hi = max(d[true].max(), d[pred].max()) + 2
    ax.plot([lo, hi], [lo, hi], "k--", lw=1)
    ax.set_xlabel("Chronological age (yr)")
    ax.set_ylabel("Predicted age (yr)")
    ax.set_title(title)
    ax.text(0.05, 0.95, f"MAE={mae:.2f}  r={r:.3f}",
            transform=ax.transAxes, va="top", fontsize=10,
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.7))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
scatter_ax(axes[0], face_ixi,  "IXI predictions (ixi_faceage_splits)", "#2196F3")
scatter_ax(axes[1], brain_ixi, "SynthBA on IXI",                        "#4CAF50")
fig.tight_layout()
fig.savefig(FIGURES / "scatter_ixi.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved scatter_ixi.png")


In [ ]:
# Face-brain gap correlation
d = gap_df[["face_age_gap", "brain_age_gap", "true_age"]].dropna()
assert len(d) > 10, f"gap_df too small after dropna: {len(d)}"

r_p, p_p = stats.pearsonr(d["face_age_gap"], d["brain_age_gap"])
r_s, p_s = stats.spearmanr(d["face_age_gap"], d["brain_age_gap"])
slope, intercept, *_ = stats.linregress(d["face_age_gap"], d["brain_age_gap"])
x_ln = np.linspace(d["face_age_gap"].min(), d["face_age_gap"].max(), 100)

fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(d["face_age_gap"], d["brain_age_gap"],
                c=d["true_age"], cmap="plasma", alpha=0.7, s=30)
ax.plot(x_ln, slope * x_ln + intercept, "k--", lw=1.5)
ax.axhline(0, color="grey", lw=0.7, ls=":")
ax.axvline(0, color="grey", lw=0.7, ls=":")
ax.set_xlabel("Face age gap (yr)")
ax.set_ylabel("Brain age gap (yr)")
ax.set_title("Face-brain age gap correlation (IXI)")
txt = (
    f"Pearson r={r_p:.2f}  p={p_p:.2e}"
    + chr(10)
    + f"Spearman r={r_s:.2f}  p={p_s:.2e}"
)
ax.text(0.05, 0.97, txt, transform=ax.transAxes, va="top", fontsize=9,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))
plt.colorbar(sc, ax=ax, label="Chronological age (yr)")
fig.tight_layout()
fig.savefig(FIGURES / "gap_correlation.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"N={len(d)}  Pearson r={r_p:.3f} p={p_p:.2e}  Spearman r={r_s:.3f} p={p_s:.2e}")


In [ ]:
# SIMON scan-rescan reproducibility
face_vals  = face_simon["predicted_age"].dropna()
brain_vals = brain_simon.loc[brain_simon["status"] == "ok", "predicted_age"].dropna()

assert len(face_vals)  > 0, "No face predictions for SIMON"
assert len(brain_vals) > 0, "No brain predictions with status=ok for SIMON"

fig, ax = plt.subplots(figsize=(7, 4))
ax.violinplot([face_vals, brain_vals], positions=[1, 2], showmedians=True)
for pos, (vals, color) in enumerate(
    [(face_vals, "#2196F3"), (brain_vals, "#4CAF50")], start=1
):
    ax.scatter([pos] * len(vals), vals, alpha=0.35, s=10, color=color, zorder=3)

ax.set_xticks([1, 2])
ax.set_xticklabels(["Face (GPA+Ridge)", "Brain (SynthBA)"])
ax.set_ylabel("Predicted age (yr)")
ax.set_title("SIMON scan-rescan: predicted age across 99 scans (1 subject)")
fig.tight_layout()
fig.savefig(FIGURES / "simon_reproducibility.png", dpi=150, bbox_inches="tight")
plt.show()

for label, vals in [("Face (GPA+Ridge)", face_vals), ("Brain (SynthBA)", brain_vals)]:
    print(f"{label}: n={len(vals)}  mean={vals.mean():.1f}  "
          f"std={vals.std():.2f}  range=[{vals.min():.1f}, {vals.max():.1f}]")
